So, since we've tokenized our data let's load it immediately and perform a simple safe check to be sure everything is as it should be.

In [55]:
from datasets import load_from_disk
from gensim.corpora import Dictionary
dataset = load_from_disk("../data/processed/tokenized_codexglue")
code_dictionary = Dictionary.load('./../data/processed/tokenized_codexglue/code_dictionary.pt')
docstring_dictionary = Dictionary.load('./../data/processed/tokenized_codexglue/docstring_dictionary.pt')

In [56]:
print(dataset)
print(code_dictionary)
print(docstring_dictionary)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 10000
    })
    valid: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 2000
    })
})
Dictionary<68072 unique tokens: ['";"', '"__"', '"s"', '(', ')']...>
Dictionary<10018 unique tokens: ['-', '.', 'QIIME', 'Return', 'a']...>


In [57]:
print(dataset['train'][0])

{'input_ids': [14, 20, 68071, 16, 6, 15, 11, 68070, 4, 10, 15, 11, 15, 5, 68069, 17, 11, 16, 7, 19, 68071, 15, 4, 18, 17, 12, 8, 13, 5, 15, 5, 17, 12, 9, 13, 7, 19, 68071, 68068, 4, 12, 8, 13], 'labels': [10017, 5, 12, 7, 9, 13, 14, 8, 4, 10016, 10014, 6, 11, 10, 10015]}


In [58]:
print(dataset['valid'][0])

{'input_ids': [14, 0, 68071, 255, 6, 42, 11, 48, 4, 10, 36, 42, 98, 48, 10, 42, 11, 41, 7, 42, 7, 100, 68071, 1374, 7, 47759, 68071, 4, 6, 0, 4, 132, 3666, 7, 30959, 68071, 4, 28, 7601, 10, 0, 68071, 41, 7, 42, 7, 100, 68071, 7601, 6, 31706, 4, 4, 0, 11, 41, 7, 42, 7, 100, 68071, 7601, 6, 0, 4, 132, 20151, 7, 18806, 68071, 0, 6, 665, 4, 28, 20159, 10, 89, 1421, 6, 6175, 6, 1586, 96, 41, 7, 6185, 68071, 7601, 4, 10, 89, 1433, 96, 1586, 10, 7391, 11, 41, 7, 42, 7, 100, 68071, 1421, 6, 1433, 4, 36, 7391, 488, 0, 10, 20159, 7, 1199, 68071, 7391, 6, 41, 7, 42, 7, 16423, 68071, 7391, 6, 7601, 4, 4, 132, 58, 68071, 0, 6, 6514, 4, 28, 523, 10, 0, 11, 523, 7, 172, 68071, 4, 132, 58, 68071, 42, 6, 6519, 4, 28, 523, 10, 0, 7, 2843, 68071, 68071, 0, 6, 255, 7, 0, 4, 6, 523, 4], 'labels': [2148, 825, 41, 4, 1852, 535, 178, 34]}


In [59]:
print(dataset['test'][0])

{'input_ids': [14, 0, 68071, 11241, 4, 10, 0, 11, 12, 13, 3941, 11, 29351, 68071, 11241, 4, 89, 679, 96, 3941, 7, 26746, 68071, 0, 4, 10, 2207, 11, 679, 7, 26746, 68071, 1163, 4, 12, 8, 13, 0, 7, 200, 68071, 2207, 7, 3940, 12, 8, 13, 7, 236, 4, 18, 0], 'labels': [670, 10014, 767, 349, 570, 1420, 41, 943, 1344, 10015, 3816, 0, 10015]}


For starters we're gonna build a simple seq2seq model, without attention (we'll then add it later). We're gonna start with a RNN, since, as explained in [this paper](https://www.arxiv.org/pdf/1909.04352v1#:~:text=Recurrent%20neural%20network%20is%20most,a%20special%20kind%20of%20RNN), RNN seems the more convenient choice between that and CNN. 

They also say that LSTM produced great results, so that's the way we're gonna take for now.

In [60]:
import torch
import torch.nn as nn

In [61]:

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding_dim = embedding_dim
        self.hidden = None # final hidden state
        self.cell = None
        self.basic_rnn = nn.LSTM(self.embedding_dim, self.hidden_dim, batch_first=True) # NLF

    def forward(self, X):
        embedded = self.embedding(X)
        batch_first_output, (self.hidden, self.cell) = self.basic_rnn(embedded) # NLH, 1NH, 1NH
        return batch_first_output, (self.hidden, self.cell)

In [62]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, bos_id, eos_id, pad_id):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding_dim = embedding_dim
        self.hidden = None
        self.cell = None
        self.bos_id = bos_id
        self.eos_id = eos_id
        self.pad_id = pad_id
        self.basic_rnn = nn.LSTM(self.embedding_dim, self.hidden_dim, batch_first=True) # NLF

    def init_hidden(self, encoder_states):
        self.hidden, self.cell = encoder_states

    def forward(self, X):
        # X is N, 1, F
        embedded = self.embedding(X)
        batch_first_output, (self.hidden, self.cell) = self.basic_rnn(embedded, (self.hidden, self.cell))
        return batch_first_output, (self.hidden, self.cell)

Now, let's just test the encoder and the decoder with some random tensors, just to ensure it all workd as expected.

In [63]:
batch_size = 2
seq_len = 10
embedding_dim = 256
hidden_dim = 512
vocab_size = 10000

torch.manual_seed(42)
X = torch.randint(vocab_size, (batch_size, seq_len))
print(X.shape) # should be NLF (2, 10, 256)
print(X[0])

torch.Size([2, 10])
tensor([7542, 6067, 6876, 6414,   26, 7335, 8620, 1924, 4950, 7113])


In [64]:
encoder = Encoder(vocab_size=vocab_size, embedding_dim=embedding_dim, hidden_dim=hidden_dim)
encoder_output, (enc_hidden, enc_cell) = encoder(X)

print(encoder_output.shape) # NLH (2, 10, 512)
print(enc_hidden.shape) # 1NH (1, 2, 512)
print(enc_cell.shape) # 1NH (1, 2, 512)

torch.Size([2, 10, 512])
torch.Size([1, 2, 512])
torch.Size([1, 2, 512])


In [ ]:
decoder = Decoder(
  vocab_size=vocab_size, 
  embedding_dim=embedding_dim, 
  hidden_dim=hidden_dim,
  bos_id=docstring_dictionary.id2token("[BOS]"),
  eos_id=docstring_dictionary.id2token("[EOS]"),
  pad_id=docstring_dictionary.id2token("[PAD]")
)
decoder.init_hidden((enc_hidden, enc_cell))
decoder_input = torch.randint(vocab_size, (batch_size, 1)) # N, 1, simulates embeddings of previous target token
decoder_output, (dec_hidden, dec_cell) = decoder(decoder_input)

print(decoder_output.shape) # N, 1, H (2, 1, 512)
print(dec_hidden.shape) # 1, N, H (1, 2, 512)
print(dec_cell.shape) # 1, N, H (1, 2, 512)

TypeError: Decoder.__init__() missing 3 required positional arguments: 'bos_id', 'eos_id', and 'pad_id'

We're now going to build the whole encoder decoder. We're gonna build it using teacher-forcing, as without it the performance of our model is probably going to be pretty poor, so it's not worth testing it.

In [ ]:
class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder, input_len, target_len, teacher_forcing_prob=0.5):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.input_len = input_len
        self.target_len = target_len
        self.teacher_forcing_prob = teacher_forcing_prob
        self.outputs = None

    def init_outputs(self, batch_size):
        device = next(self.parameters()).device
        # N, L, V (since output is logits)
        self.outputs = torch.zeros(batch_size,
                              self.target_len,
                              self.decoder.vocab_size).to(device)

    def store_output(self, i, out):
        # Stores the output
        self.outputs[:, i:i+1, :] = out

    def forward(self, source_seq, target_seq):
        # the target seq will be empty in testing mode
        # N, L, F
        self.init_outputs(source_seq.shape[0])

        encoder_outputs, (enc_hidden, enc_cell) = self.encoder(source_seq)
        # Output is NLH, 1NH, 1NH
        self.decoder.init_hidden((enc_hidden, enc_cell))

        # First input to the decoder is the <BOS> token
        bos_id = self.decoder.bos_id
        dec_inputs = torch.full(
            (batch_size, 1),
            bos_id,
            dtype=torch.long,
            device=source_seq.device
        )
        

        # Generates as many outputs as the target length
        for i in range(self.target_len):
            # Output of decoder is N, 1, F
            out = self.decoder(dec_inputs)
            self.store_output(i, out)

            prob = self.teacher_forcing_prob
            # In evaluation/test the target sequence is
            # unknown, so we cannot use teacher forcing
            if not self.training:
                prob = 0

            # If it is teacher forcing
            if torch.rand(1) <= prob:
                # Takes the actual element
                dec_inputs = target_seq[:, i:i+1]
            else:
                # Takes the predicted element
                next_tokens = out.argmax(dim=-1)  # logits to token ids
                dec_inputs = next_tokens

        return self.outputs